# Warm-Up: Your First Quantum Programs

**Everyone starts here.** We go through this notebook together at the start of the hackathon (about 45 minutes). After it, your team picks a challenge.

**How notebooks work.** A notebook is a stack of cells. Click a code cell and press **Shift + Enter** to run it. Run the cells from top to bottom. If things get weird, restart (*Kernel → Restart*, or *Runtime → Restart session* in Colab) and run from the top again.

**How missions work.** Each mission has a code cell with a `# TODO`. Edit the code, run it, then run the check cell under it. A ✅ means you got it. A ❌ comes with a hint.

In [ ]:
# Using Google Colab? Delete the "#" at the start of the next line, then run this cell once.
# !pip install -q "qiskit[visualization]"

In [ ]:
# SETUP: run this cell first (click it, then press Shift+Enter).
# You don't need to read or change anything in this cell.

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import math, random, qiskit

_sampler = StatevectorSampler()

def _needs_measure(qc):
    if qc.num_clbits == 0:
        raise ValueError("Your circuit has no measurements yet. Add qc.measure_all() as the last step of your circuit.")

def run(qc, shots=1000):
    """Run a circuit many times. Returns counts, e.g. {'0': 507, '1': 493}."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_counts()

def run_once(qc):
    """Run a circuit one time. Returns a single result as a string, e.g. '101'."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=1).result()[0].join_data().get_bitstrings()[0]

def run_many(qc, shots):
    """Run a circuit `shots` times. Returns every result in order, e.g. ['0', '1', '1', ...]."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_bitstrings()

def probabilities(qc):
    """Exact chance of each result, with no randomness. Measurements at the end are ignored."""
    sv = Statevector(qc.remove_final_measurements(inplace=False))
    return {str(k): round(float(v), 4) for k, v in sv.probabilities_dict().items() if v > 1e-9}

def _close(got, want, tol=0.02):
    return all(abs(got.get(k, 0) - want.get(k, 0)) <= tol for k in set(got) | set(want))

def _report(ok, yes, hint):
    if ok:
        print("\u2705 " + yes)
    else:
        print("\u274c Not yet. Hint: " + hint)

def _check_probs(qc, want, yes, hint):
    try:
        ok = _close(probabilities(qc), want)
    except Exception:
        ok = False
    _report(ok, yes, hint)

# ---- checks for this notebook ----
def check_mission_1(qc):
    _check_probs(qc, {"1": 1.0}, "Mission 1 complete: your qubit always reads 1.",
                 "add qc.x(0) before qc.measure_all().")

def check_mission_2(qc):
    _check_probs(qc, {"0": 0.5, "1": 0.5}, "Mission 2 complete: a fair quantum coin!",
                 "add qc.h(0) before qc.measure_all().")

def check_mission_3(qc):
    ok = qc.count_ops().get("h", 0) == 2
    try:
        ok = ok and _close(probabilities(qc), {"0": 1.0})
    except Exception:
        ok = False
    _report(ok, "Mission 3 complete: two H gates cancel out. That's interference!",
            "use exactly two qc.h(0) lines, then qc.measure_all().")

def check_mission_4(qc):
    _check_probs(qc, {"10": 1.0}, "Mission 4 complete: you can read results right to left.",
                 "the LEFT digit is qubit 1, so flip qubit 1 with qc.x(1).")

def check_mission_5(qc):
    _check_probs(qc, {"00": 0.5, "11": 0.5}, "Mission 5 complete: you entangled two qubits!",
                 "qc.h(0) first, then qc.cx(0, 1).")

print(f"\u2705 Setup complete (Qiskit {qiskit.__version__}). You're ready to go!")

## 1. Bits vs. qubits

A normal computer **bit** is always 0 or 1, like a coin lying flat on a table: heads or tails.

A **qubit** can be in a **superposition**, a bit like a coin spinning in the air. While it spins, it is not heads or tails yet. When you **measure** it (catch the coin), you get 0 or 1 at random, and the odds depend on how it was spinning.

Every quantum program follows three steps:

1. Start with qubits. They always start as 0.
2. Apply **gates** that change the qubits.
3. **Measure** to turn the qubits into normal 0s and 1s.

In [ ]:
# Step 1: make a circuit with 1 qubit
qc = QuantumCircuit(1)

# Step 2: no gates yet!

# Step 3: measure
qc.measure_all()

qc.draw("mpl")

Now run that circuit on the simulator 1,000 times. `run()` counts how often each result came up.

In [ ]:
counts = run(qc)
print(counts)
plot_histogram(counts)

Always 0, because nothing changed the qubit. Time to change it.

## Mission 1: Flip the bit

The **X gate** flips a qubit: 0 becomes 1 and 1 becomes 0. It is the quantum version of NOT. The code is `qc.x(0)`, where `0` means "qubit number 0."

**TODO:** Add an X gate so the result is always 1.

In [ ]:
qc = QuantumCircuit(1)
# TODO: add an X gate on qubit 0

qc.measure_all()

print(run(qc))
qc.draw("mpl")

In [ ]:
check_mission_1(qc)

## Mission 2: The quantum coin

The **H gate** (Hadamard gate) puts a qubit into an equal superposition: the spinning coin. Measure it and you get 0 half the time and 1 half the time.

**TODO:** Use `qc.h(0)` to make a fair quantum coin.

In [ ]:
qc = QuantumCircuit(1)
# TODO: add an H gate on qubit 0

qc.measure_all()

counts = run(qc)
print(counts)
plot_histogram(counts)

In [ ]:
check_mission_2(qc)

That randomness is not a trick. Normal computers fake randomness with math formulas, but a measured qubit is random by the laws of physics. Flip your coin ten times, one at a time:

In [ ]:
for flip in range(10):
    print(run_once(qc), end=" ")

## Mission 3: Un-spin the coin

**Predict first.** If you apply H twice in a row, do you get 50/50 again? Type your guess on the line below before you run anything.

My guess:

**TODO:** Build a circuit with two H gates in a row on qubit 0, then measure.

In [ ]:
qc = QuantumCircuit(1)
# TODO: two H gates on qubit 0

qc.measure_all()

print(run(qc))

In [ ]:
check_mission_3(qc)

Surprise: two H gates bring the qubit back to 0 every time. Flipping a normal coin twice could never do that.

This is **interference**. The two ways of reaching 1 cancel each other out, the way two water waves can cancel. Interference is the secret ingredient in the most powerful quantum algorithms.

## A picture of one qubit: the Bloch sphere

Physicists draw a qubit as an arrow on a sphere. Arrow up means 0, arrow down means 1, and an arrow on the equator means a 50/50 superposition. Run this to see the arrow after one H gate. Then change `qc.h(0)` to `qc.x(0)`, or delete it, and run again.

(There's no measurement here, because measuring would make the arrow snap to up or down.)

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
plot_bloch_multivector(Statevector(qc))

## Mission 4: Two qubits, read right to left

With 2 qubits there are 4 possible results: `00`, `01`, `10`, and `11`.

**Important:** Qiskit writes qubit 0 on the **right**. So `'01'` means qubit 0 is 1 and qubit 1 is 0. Run this example to see it:

In [ ]:
qc = QuantumCircuit(2)
qc.x(0)            # flip qubit 0
qc.measure_all()
print(run(qc))     # prints {'01': 1000}: qubit 0 is the RIGHT digit

**TODO:** Make a circuit whose result is always `'10'`.

In [ ]:
qc = QuantumCircuit(2)
# TODO: which qubit do you need to flip?

qc.measure_all()
print(run(qc))

In [ ]:
check_mission_4(qc)

## Mission 5: Entanglement

The **CNOT gate**, written `qc.cx(control, target)`, flips the target qubit only when the control qubit is 1.

Put the control qubit into superposition first and something strange happens: the two qubits become **entangled**. Each one is random on its own, yet they always match.

**TODO:** Apply H to qubit 0, then a CNOT with qubit 0 as the control and qubit 1 as the target.

In [ ]:
qc = QuantumCircuit(2)
# TODO: H on qubit 0, then CNOT from qubit 0 to qubit 1

qc.measure_all()

counts = run(qc)
print(counts)
plot_histogram(counts)

In [ ]:
check_mission_5(qc)

You only ever see `00` and `11`, never `01` or `10`. Check one qubit and you instantly know the other, even though neither had a value until you looked. Einstein called this "spooky action at a distance." This pair is called a **Bell pair**, and several challenges build on it.

## Cheat sheet

| Code | What it does |
| --- | --- |
| `qc = QuantumCircuit(n)` | Make a circuit with `n` qubits, all starting at 0 |
| `qc.x(q)` | Flip qubit `q` (0 ↔ 1) |
| `qc.h(q)` | Put qubit `q` into a 50/50 superposition, or undo one |
| `qc.z(q)` | Flip the hidden "phase" of qubit `q`. You can't see it by measuring right away, but it changes interference |
| `qc.ry(angle, q)` | Tilt qubit `q` by an angle, which sets the odds of getting 1 |
| `qc.cx(c, t)` | CNOT: flip qubit `t` if qubit `c` is 1. Makes entanglement |
| `qc.cz(c, t)` | Flip the phase only when both qubits are 1 |
| `qc.measure_all()` | Measure every qubit. Put this last |
| `run(qc)` | Run 1,000 times and count the results |
| `run_once(qc)` | Run once and get one result, like `'01'` |
| `run_many(qc, n)` | Run `n` times and get a list of every result |
| `probabilities(qc)` | Exact odds of each result, with no randomness |
| `qc.draw("mpl")` | Draw the circuit |
| `plot_histogram(counts)` | Bar chart of results |

Results read **right to left**: qubit 0 is the rightmost digit.

## Pick your challenge

| # | Challenge | Difficulty | You'll build |
| --- | --- | --- | --- |
| 1 | Quantum Dice Lab | ★☆☆ | Coins, dice, a loaded coin, and a fortune teller powered by true randomness |
| 2 | Quantum Coin Duel | ★☆☆ | A game where the quantum player wins every time, until you find the counter-move |
| 3 | Quantum Spy Network | ★★☆ | A quantum encryption system that catches eavesdroppers |
| 4 | Spooky Telepathy | ★★☆ | Entangled pairs that send two bits with one qubit, then a whole text message |
| 5 | Quantum Treasure Hunt | ★★★ | Grover's search: find the hidden treasure in one try |
| 6 | Quantum Art Studio | ★☆☆ | Generative art painted by superposition and entanglement |

You can also pitch your own idea. Ask the organizer first.